In [4]:
"""
Build PyTorch Geometric graphs from training data + reference instance files.

Each training JSON contains ALNS solution outcome for a given graph instance. 
For each instance we merge node-level features (x, y, demand, release_time, deadline, prize, penalty) from the matching
C/R/RC reference file, and derive `nodes_served` as the label for node selection.

The output is a list of `torch_geometric.data.Data` objects, ready for GAT training.
"""

import json
from pathlib import Path
from typing import Dict, List, Optional

import torch
from torch_geometric.data import Data

'''
TRAINING_DIR is the directory where JSON files exist. 
Each JSON file has the solution of an instance along with information regarding the global variables 
such as vehicle capacity and battery range. 
Each instance shall also include the IDs of selected nodes, objective function value, and reference information
to connect back to the graph features. 

REFERENCE_DIR is the directory where information about instances exist. 
Each JSON file has graph instances with at least the following information: 
instance id and nodes. nodes is also a dictionary storing features of each node such as prize, penalty,
(x,y) coordinate, release time, deadline,... C_instances.JSON, R_instances.JSON, RC_instances.JSON

Note that the vehicle capacity and the driving range values are pulled from JSON files of TRAINING_DIR not REFERENCE_DIR.
'''
TRAINING_DIR = Path(r"C:\Users\falkaabneh\Documents\KAIST\training data")
REFERENCE_DIR = Path(r"C:\Users\falkaabneh\Documents\KAIST\data generation")       
OUTPUT_PATH = Path("graphs.pt")                   # cached PyG dataset

NUM_CUSTOMERS = 100                               # customers are nodes 1..100, depot is node 0


# ---------------------------------------------------------------------------
# Reference loader
# ---------------------------------------------------------------------------
def load_reference_instances(reference_dir: Path) -> Dict[str, dict]:
    """Load C/R/RC reference files into a single dict keyed by instance_id."""
    reference: Dict[str, dict] = {}
    for type_name in ("C", "R", "RC"):
        filepath = reference_dir / f"{type_name}_instances.json"
        with open(filepath, "r") as f:
            instances = json.load(f)
        for inst in instances:
            reference[inst["instance_id"]] = inst
    print(f"Loaded {len(reference)} reference instances")
    return reference




def build_graph(
    instance: dict,
    reference_data: Dict[str, dict],
    vehicle_load_override: Optional[float] = None,
) -> Data:
    """
    Build a PyG Data object for graph-level regression on objective_value.

    Nodes:  depot (index 0) + served customers (indices 1..K)
    x:      [K+1, 6]  -> [x_coord, y_coord, demand, release_time, deadline, tw_width]
    edges:  complete graph, no self-loops
    edge_attr: [E, 2] -> [distance, distance / battery_capacity]
    y:      [1]       -> objective_value
    """
    instance_id = instance["instance_id"]
    if instance_id not in reference_data:
        raise KeyError(f"instance_id {instance_id!r} not found in reference data")
    ref = reference_data[instance_id]

    # ---- served customer ids (depot is handled separately) ----------------
    excluded = set(instance["customers_not_selected_first_place"]) | set(
        instance["customers_not_visited_from_selected_pool"]
    )
    nodes_served = [i for i in range(1, NUM_CUSTOMERS + 1) if i not in excluded]
    if not nodes_served:
        raise ValueError(f"{instance_id}: no served customers")

    # Lookup: node_id -> node dict from the reference file
    ref_nodes_by_id = {n["node_id"]: n for n in ref["nodes"]}

    # Graph order: depot first, then served customers in ascending id
    included_ids = [0] + nodes_served

    # ---- node features (K+1 x 6) ------------------------------------------
    features = []
    for nid in included_ids:
        node = ref_nodes_by_id[nid]
        rt = float(node["release_time"])
        dl = float(node["deadline"])
        features.append([
            float(node["x"]),
            float(node["y"]),
            float(node["demand"]),
            rt,
            dl,
            dl - rt,                       # time-window width
        ])
    x = torch.tensor(features, dtype=torch.float)        # [K+1, 6]

    # ---- battery capacity (needed for edge features) ----------------------
    bat_raw = instance.get("battery_capacity")
    if bat_raw is None:
        raise ValueError(f"{instance_id}: battery_capacity missing (needed for edge features)")
    bat = float(bat_raw)

    # ---- complete graph (no self-loops) -----------------------------------
    num_nodes = x.size(0)
    row, col = torch.meshgrid(
        torch.arange(num_nodes), torch.arange(num_nodes), indexing="ij"
    )
    mask = row != col
    edge_index = torch.stack([row[mask], col[mask]], dim=0)  # [2, E]

    # ---- edge features: [distance, distance / battery_capacity] -----------
    coords = x[:, :2]
    diff = coords[edge_index[0]] - coords[edge_index[1]]
    dist = diff.norm(dim=1, keepdim=True)                # [E, 1]
    range_ratio = dist / bat                             # [E, 1]
    edge_attr = torch.cat([dist, range_ratio], dim=1)    # [E, 2]

    # ---- regression target ------------------------------------------------
    obj = float(instance.get("objective_value", float("nan")))
    y = torch.tensor([obj], dtype=torch.float)           # [1]

    data = Data(x=x, edge_index=edge_index, edge_attr=edge_attr, y=y)

    # ---- graph-level scalars as tensors (batch correctly) -----------------
    # The following few lines handle cases where JSON files do not have vehicle_load_capacity feature/field value 
    if vehicle_load_override is not None:
        vlc = vehicle_load_override
    else:
        vlc = float(instance.get("vehicle_load_capacity", 50.0))
    data.battery_capacity      = torch.tensor([bat], dtype=torch.float)
    data.vehicle_load_capacity = torch.tensor([vlc], dtype=torch.float)

    # ---- string / list metadata (not used by the model) -------------------
    data.instance_id      = instance_id
    data.instance_type    = instance["instance_type"]
    data.nodes_served     = nodes_served       # original customer ids
    data.original_node_ids = included_ids      # depot=0 then served, index-aligned with x

    return data


def load_all_graphs(
    training_dir: Path = TRAINING_DIR,
    reference_dir: Path = REFERENCE_DIR,
) -> List[Data]:
    reference_data = load_reference_instances(reference_dir)

    json_files = sorted(training_dir.glob("*.json")) + sorted(training_dir.glob("*.JSON"))
    json_files = list(dict.fromkeys(json_files))   # dedupe in case both cases match
    print(f"Found {len(json_files)} training JSON files in {training_dir}")

    graphs: List[Data] = []
    skipped = 0
    for filepath in json_files:
        # Filename (without extension) ending in "battery" → force vehicle_load_capacity = 50.0
        override = 50.0 if filepath.stem.lower().endswith("battery") else None
        if override is not None:
            print(f"  {filepath.name}: forcing vehicle_load_capacity = 50.0")

        with open(filepath, "r") as f:
            instances = json.load(f)
        for instance in instances:
            try:
                graphs.append(build_graph(instance, reference_data, vehicle_load_override=override))
            except Exception as e:
                skipped += 1
                print(f"  skipped {instance.get('instance_id', '?')}: {e}")

    print(f"Built {len(graphs)} graphs ({skipped} skipped)")
    return graphs


# ---------------------------------------------------------------------------
# CLI entry point
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    graphs = load_all_graphs()

    # Quick sanity check on the first graph
    if graphs:
        g = graphs[0]
        print("\nFirst graph summary:")
        print(f"  instance_id      = {g.instance_id}")
        print(f"  instance_type    = {g.instance_type}")
        print(f"  x.shape          = {tuple(g.x.shape)}")
        print(f"  edge_index.shape = {tuple(g.edge_index.shape)}")
        print(f"  edge_attr.shape  = {tuple(g.edge_attr.shape)}")
        print(f"  objective_value  = {g.y}")
        print(f"  battery_capacity = {g.battery_capacity}")
        print(f"  vehicle_load_cap = {g.vehicle_load_capacity}")

    torch.save(graphs, OUTPUT_PATH)
    print(f"\nSaved {len(graphs)} graphs to {OUTPUT_PATH}")

Loaded 100 reference instances
Found 31 training JSON files in C:\Users\falkaabneh\Documents\KAIST\training data
  experiment_results_C_11_20_loose_battery.json: forcing vehicle_load_capacity = 50.0
  experiment_results_C_1_10_loose_battery.json: forcing vehicle_load_capacity = 50.0
  experiment_results_C_21_30_loose_battery.json: forcing vehicle_load_capacity = 50.0
  experiment_results_R_11_20_loose_battery.json: forcing vehicle_load_capacity = 50.0
  experiment_results_R_1_10_loose_battery.json: forcing vehicle_load_capacity = 50.0
  experiment_results_R_21_30_loos_battery.json: forcing vehicle_load_capacity = 50.0
  experiment_results_RC_11_20_loose_battery.json: forcing vehicle_load_capacity = 50.0
  experiment_results_RC_1_10_loose_battery.json: forcing vehicle_load_capacity = 50.0
  experiment_results_RC_21_30_loose_battery.json: forcing vehicle_load_capacity = 50.0
  experiment_results_RC_30_40_loose_battery.json: forcing vehicle_load_capacity = 50.0
Built 20029 graphs (0 skipp

In [8]:
graphs = torch.load("graphs.pt", weights_only=False)

tensor([-2500.3352])

In [16]:
graphs = torch.load("graphs.pt", weights_only=False)
g = graphs[0]
print(g)
print("served nodes:", len(g.nodes_served), "x.shape:", g.x.shape)
print("edge_attr sample:", g.edge_attr[:3])
print("y:", g.y.item(), "battery:", g.battery_capacity.item(), "load:", g.vehicle_load_capacity.item())

Data(x=[81, 6], edge_index=[2, 6480], edge_attr=[6480, 2], y=[1], battery_capacity=[1], vehicle_load_capacity=[1], instance_id='R_001', instance_type='R', nodes_served=[80], original_node_ids=[81])
served nodes: 80 x.shape: torch.Size([81, 6])
edge_attr sample: tensor([[10.6021,  0.0901],
        [29.8171,  0.2533],
        [31.0299,  0.2636]])
y: -2500.335205078125 battery: 117.72547912597656 load: 50.0
